# Learning notebook

These working notes retain the sequence of experiments and diagnostics.
Model imports now use the current 32/16 implementation, so rerunning old
experiment cells does not reconstruct earlier architectures. Several cells
train for 50,000 epochs. For the final reproducible workflow use
[finalExperiment.ipynb](finalExperiment.ipynb); published scores and their
limitations are recorded in [the findings](../docs/FINAL_FINDINGS.md).

Dataset paths are checkout-relative. Features and targets come from an
explicit validated `prj` join; raw CSV row order is not assumed to match.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from circuitcost.data import DATASET_SUBDIR, find_project_root, load_designs

project_root = find_project_root()

In [ ]:
data_dir = project_root / DATASET_SUBDIR

hls = pd.read_csv(data_dir / "post_hls_info.csv")
impl = pd.read_csv(data_dir / "post_implementation_info.csv")

# Explicit one-to-one prj join, with checks for unmatched and duplicate IDs.
# The loader preserves HLS file order so the recorded random split is reproducible.
designs = load_designs(data_dir)
display(designs.head())

In [ ]:
#Preparing features from HLS table
feature_columns = [
    "clk_target(ns)",
    "clk_estimated(ns)",
    "clk_uncertainty(ns)",
    "bram",
    "dsp",
    "ff",
    "lut",
    "c_num_arith",
    "c_num_logic",
    "rtl_num_arith",
    "rtl_num_logic"
]

In [ ]:
#Defining X and y
X = designs[feature_columns].copy()
y = designs["Total LUTs"].copy()
X.head()

In [ ]:
y.head()

Install dependencies before opening the notebook: run `make setup` and `make notebooks` from the project root. Select that environment as the kernel.

In [ ]:
#Splitting the Dataset into Training,Validation and Testing
from sklearn.model_selection import train_test_split

#First Split : 70% Training, 30% remaining
X_train, X_remaining, y_train, y_remaining = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

In [ ]:
#Second split: divide the remaining 30% equally
X_val, X_test, y_val, y_test = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.50,
    random_state=42
)

In [ ]:
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

In [ ]:
#Normalizing the values of dataset
#xscaled = x-training mean/ training standard deviation

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print("Training means:", X_train_scaled.mean(axis=0))
print("Training standard deviations:", X_train_scaled.std(axis=0))

In [ ]:
#Checking the scaled values of training, validation and test dataset

print("Training Set Values:",X_train_scaled[:5])
print("Validation Set Values:",X_val_scaled[:5])
print("Test Set Values:",X_test_scaled[:5])

In [ ]:
#Establishing a baseline
#Baseline 1: HLS estimate , Prediction rule = Predicted implementation LUTs=HLS lut
#Baseline 2: Linear regression, Prediction rule = Weighted combination of selected HLS feature

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pred_hls_val = X_val["lut"].to_numpy()

In [ ]:
#Train Linear regression

linear_model = LinearRegression() #Creates an unfitted model
linear_model.fit(X_train_scaled, y_train) #Learns a coefficient for each feature and an intercept from the training example
pred_linear_val = linear_model.predict(X_val_scaled) #applies the learned relationship to validation inputs

In [ ]:
#Evaluate both baselines consistently

def evaluate_predictions(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true,y_pred),
        "RMSE": mean_squared_error(y_true,y_pred) ** 0.5,
        "R2": r2_score(y_true,y_pred)
    }

In [ ]:
#Check the result
results = pd.DataFrame({
    "HLS estimate" : evaluate_predictions(y_val, pred_hls_val),
    "Linear Regression" : evaluate_predictions(y_val,pred_linear_val),
}).T

display(results)

In [ ]:
comparison = pd.DataFrame({
    "Actual LUTs": y_val,
    "HLS prediction": pred_hls_val,
    "Linear prediction": pred_linear_val,
})

display(comparison.head())

In [ ]:
import sys
print(sys.executable)

Install dependencies before opening the notebook: run `make setup` and `make notebooks` from the project root. Select that environment as the kernel.

In [ ]:
#using custom MLP
from circuitcost.models import NeuralNetworks
from circuitcost.train import train_model

np.random.seed(42)

model = NeuralNetworks(n_inputs=X_train_scaled.shape[1])

history = train_model(
    model,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    epochs=1000,
    learning_rate=0.000001,
)

In [ ]:
#Plotting the learining curves
import matplotlib.pyplot as plt

plt.plot(history["train_loss"], label="Training")
plt.plot(history["val_loss"], label="Validation")

plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("Neural network learning curves")
plt.legend()
plt.show()

In [ ]:
#Generate validation predictions

pred_nn_val = model.forward(X_val_scaled).ravel()

print("Predictions are finite:", np.isfinite(pred_nn_val).all())
print("Negative predictions:", np.sum(pred_nn_val < 0))

In [ ]:
#Calculate same metrics as baseline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

mae_nn = mean_absolute_error(y_val, pred_nn_val)
rmse_nn = np.sqrt(mean_squared_error(y_val, pred_nn_val))
r2_nn = r2_score(y_val, pred_nn_val)

print(f"Neural network MAE:  {mae_nn:.2f} LUTs")
print(f"Neural network RMSE: {rmse_nn:.2f} LUTs")
print(f"Neural network R²:   {r2_nn:.4f}")

In [ ]:
#inspect indivual predictions


comparison = pd.DataFrame({
    "Actual LUTs": np.asarray(y_val),
    "Predicted LUTs": pred_nn_val,
})

comparison["Absolute error"] = (
    comparison["Predicted LUTs"] - comparison["Actual LUTs"]
).abs()

display(comparison.sort_values("Absolute error", ascending=False).head(10))

In [ ]:
# Use the already-trained models
pred_linear_val = linear_model.predict(X_val_scaled)
pred_nn_val = model.forward(X_val_scaled).ravel()

results = pd.DataFrame({
    "HLS estimate": evaluate_predictions(
        y_val, X_val["lut"].to_numpy()
    ),
    "Linear regression": evaluate_predictions(
        y_val, pred_linear_val
    ),
    "Custom neural network": evaluate_predictions(
        y_val, pred_nn_val
    ),
}).T

display(results.round(3))

In [ ]:
train_rmse = np.sqrt(history["train_loss"][-1])
val_rmse = np.sqrt(history["val_loss"][-1])

print(f"Final training RMSE:   {train_rmse:.2f} LUTs")
print(f"Final validation RMSE: {val_rmse:.2f} LUTs")

print(f"Validation − training: {val_rmse - train_rmse:.2f} LUTs")

In [ ]:
#Model does not woek Properly
#Adding following steps to analyze

In [ ]:
import copy

Xtr = np.asarray(X_train_scaled, dtype=float)
ytr = np.asarray(y_train, dtype=float).reshape(-1,1)

assert designs.index.is_unique  # Targets were matched by prj
assert X_train.index.equals(y_train.index)
assert len(Xtr) == len(ytr)

print("Training features:", Xtr.shape)
print("Training targets:", ytr.shape)
print("Features finite:", np.isfinite(Xtr).all())
print("Targets finite:", np.isfinite(ytr).all())

In [ ]:
#Inspect learning over time
train_rmse_curve = np.sqrt(np.asarray(history["train_loss"]))
val_rmse_curve = np.sqrt(np.asarray(history["val_loss"]))

plt.plot(train_rmse_curve, label="Training")
plt.plot(val_rmse_curve, label="Validation")

plt.xlabel("Epoch")
plt.ylabel("RMSE (LUTs)")
plt.legend()
plt.show()

print("Final training RMSE:", train_rmse_curve[-1])
print("Final validation RMSE:", val_rmse_curve[-1])

In [ ]:
#Check that gradients and updates exist
probe = copy.deepcopy(model)
from circuitcost.train import mse_gradient


predictions = probe.forward(Xtr[:8])
gradient = mse_gradient(ytr[:8], predictions)
probe.backward(gradient)

dense_layers = [
    layer for layer in probe.layers
    if hasattr(layer, "weights")
]

for number, layer in enumerate(dense_layers, start=1):
    print(
        f"Layer {number}:",
        "weight-gradient size =", np.linalg.norm(layer.dweights),
        "gradients finite =", np.isfinite(layer.dweights).all(),
    )

before = dense_layers[0].weights.copy()
probe.step(learning_rate=1e-6)

print(
    "First-layer weight change:",
    np.linalg.norm(dense_layers[0].weights - before),
)

In [ ]:
#Comparing the update size with weight size
weight_norm = np.linalg.norm(before)

update_norm = np.linalg.norm(
    dense_layers[0].weights - before
)

relative_update = update_norm / (weight_norm + 1e-12)

print("Original weight norm:", weight_norm)
print("Update norm:", update_norm)
print(f"Relative update: {relative_update:.2%}")

In [ ]:
#Checking whether that update improved the loss
from circuitcost.train import mse_loss

xb = Xtr[:8]
yb = ytr[:8]

loss_before = mse_loss(yb, model.forward(xb))
loss_after = mse_loss(yb, probe.forward(xb))

print("Before update:", loss_before)
print("After update:", loss_after)

In [ ]:
predictions = probe.forward(Xtr)
gradient = mse_gradient(ytr, predictions)

In [ ]:
#Training session 2
np.random.seed(42)

longer_model = NeuralNetworks(
    n_inputs=X_train_scaled.shape[1]
)

longer_history = train_model(
    longer_model,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    epochs=50000,
    learning_rate=0.000001,
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    np.sqrt(longer_history["train_loss"]),
    label="Training: longer run",
    color="tab:blue",
    alpha=0.5,
    linewidth=0.7,
)

plt.plot(
    np.sqrt(longer_history["val_loss"]),
    label="Validation: longer run",
    color="tab:orange",
    alpha=0.5,
    linewidth=0.7,
)

plt.xlabel("Epoch")
plt.ylabel("RMSE (LUTs)")
plt.legend()
plt.show()

In [ ]:
train_mse = longer_history["train_loss"][-1]
val_mse = longer_history["val_loss"][-1]

print(f"Final training MSE:   {train_mse:.2f}")
print(f"Final validation MSE: {val_mse:.2f}")

print(f"Final training RMSE:   {np.sqrt(train_mse):.2f} LUTs")
print(f"Final validation RMSE: {np.sqrt(val_mse):.2f} LUTs")

In [ ]:
best_index = np.argmin(longer_history["val_loss"])
best_mse = longer_history["val_loss"][best_index]

print("Best validation epoch:", best_index + 1)
print(f"Best validation RMSE: {np.sqrt(best_mse):.2f} LUTs")

In [ ]:
print("Selected epoch:", longer_history["best_epoch"])

print(
    "Selected validation RMSE:",
    np.sqrt(longer_history["best_val_loss"]),
)

In [ ]:
pred_nn_val = longer_model.forward(X_val_scaled).ravel()

results = pd.DataFrame({
    "Linear regression": evaluate_predictions(
        y_val, pred_linear_val
    ),
    "Custom MLP": evaluate_predictions(
        y_val, pred_nn_val
    ),
}).T

display(results)

In [ ]:
#Check how errors are distributed
actual = np.asarray(y_val).ravel()

absolute_errors = pd.DataFrame({
    "Linear regression": np.abs(
        np.asarray(pred_linear_val).ravel() - actual
    ),
    "Custom MLP": np.abs(
        np.asarray(pred_nn_val).ravel() - actual
    ),
})

display(
    absolute_errors.describe(
        percentiles=[0.50, 0.90, 0.95]
    ).T
)

In [ ]:
#checking how oftern MLP performs better
mlp_wins = (
    absolute_errors["Custom MLP"]
    < absolute_errors["Linear regression"]
)

print(f"MLP has lower error on {mlp_wins.mean():.1%} of designs")

In [ ]:
#Checking where each model wins
diagnostics = absolute_errors.copy()

diagnostics["Actual LUTs"] = np.asarray(y_val).ravel()

diagnostics["MLP advantage"] = (
    diagnostics["Linear regression"]
    - diagnostics["Custom MLP"]
)

# Designs where MLP improves the most
display(
    diagnostics.sort_values(
        "MLP advantage", ascending=False
    ).head(10)
)

# Designs where MLP performs worse
display(
    diagnostics.sort_values(
        "MLP advantage", ascending=True
    ).head(10)
)

In [ ]:
#updated architecture
import importlib
import numpy as np
import pandas as pd

import circuitcost.models as models
import circuitcost.train as training

importlib.reload(models)
importlib.reload(training)

print("Loaded model module:", models.__file__)
print("Loaded training module:", training.__file__)

In [ ]:
np.random.seed(42)

trial_model = models.NeuralNetworks(
    n_inputs=X_train_scaled.shape[1]
)

for layer in trial_model.layers:
    if hasattr(layer, "weights"):
        print("Weights:", layer.weights.shape)

print(
    "Prediction shape:",
    trial_model.forward(X_train_scaled[:2]).shape
)

In [ ]:
trial_history = training.train_model(
    trial_model,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    epochs=50000,
    learning_rate=0.000001,
)

print("Selected epoch:", trial_history["best_epoch"])

print(
    "Selected validation RMSE:",
    np.sqrt(trial_history["best_val_loss"])
)

In [ ]:
trial_predictions = trial_model.forward(
    X_val_scaled
).ravel()

linear_predictions = linear_model.predict(X_val_scaled)

trial_results = pd.DataFrame({
    "Linear regression": evaluate_predictions(
        y_val, linear_predictions
    ),
    "New MLP": evaluate_predictions(
        y_val, trial_predictions
    ),
}).T

display(trial_results)

In [ ]:
#Preserve the successful model
import copy
import numpy as np
import pandas as pd

candidate_model = copy.deepcopy(trial_model)
candidate_scaler = copy.deepcopy(scaler)
candidate_linear = copy.deepcopy(linear_model)
candidate_features = list(feature_columns)

candidate_predictions = candidate_model.forward(
    X_val_scaled
).ravel()

candidate_scores = evaluate_predictions(
    y_val, candidate_predictions
)

candidate_epoch = int(trial_history["best_epoch"])

print("Selected epoch:", candidate_epoch)
print("Validation metrics:", candidate_scores)

assert np.isclose(
    candidate_scores["RMSE"] ** 2,
    trial_history["best_val_loss"],
)

In [ ]:
#Save the parameters, preproscessing and configuration
import json
import joblib
from pathlib import Path

project_root = Path(models.__file__).resolve().parents[2]
artifact_dir = project_root / "artifacts" / "mlp_32_16_seed42"
artifact_dir.mkdir(parents=True, exist_ok=True)

parameters = {}

for i, layer in enumerate(
    layer for layer in candidate_model.layers
    if hasattr(layer, "weights")
):
    parameters[f"weights_{i}"] = layer.weights.copy()
    parameters[f"biases_{i}"] = layer.biases.copy()

np.savez_compressed(
    artifact_dir / "weights.npz",
    **parameters,
)

joblib.dump(candidate_scaler, artifact_dir / "scaler.joblib")
joblib.dump(candidate_linear, artifact_dir / "linear_model.joblib")

metadata = {
    "features": candidate_features,
    "hidden_layers": [32, 16],
    "target": "Total LUTs",
    "target_scaling": "none",
    "seed": 42,
    "learning_rate": 0.000001,
    "max_epochs": 50000,
    "best_epoch": candidate_epoch,
    "split_random_state": 42,
    "validation_metrics": {
        name: float(value)
        for name, value in candidate_scores.items()
    },
    "split_design_ids": {
        name: frame.index.tolist()
        for name, frame in [
            ("train", X_train),
            ("validation", X_val),
            ("test", X_test),
        ]
    },
}

(artifact_dir / "metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Saved to:", artifact_dir)

In [ ]:
#Check whether other initalization seeds also work

seed_records = [{
    "seed": 42,
    "status": "ok",
    "best_epoch": candidate_epoch,
    **candidate_scores,
}]

for seed in [7, 21]:
    np.random.seed(seed)

    repeat_model = models.NeuralNetworks(
        n_inputs=X_train_scaled.shape[1]
    )

    try:
        repeat_history = training.train_model(
            repeat_model,
            X_train_scaled,
            y_train,
            X_val_scaled,
            y_val,
            epochs=50000,
            learning_rate=0.000001,
        )

        predictions = repeat_model.forward(
            X_val_scaled
        ).ravel()

        seed_records.append({
            "seed": seed,
            "status": "ok",
            "best_epoch": repeat_history["best_epoch"],
            **evaluate_predictions(y_val, predictions),
        })

    except ValueError as error:
        seed_records.append({
            "seed": seed,
            "status": "failed",
            "error": str(error),
        })

seed_results = pd.DataFrame(seed_records)
display(seed_results)

successful = seed_results[
    seed_results["status"] == "ok"
]

display(
    successful[["MAE", "RMSE", "R2"]].agg(["mean", "std"])
)

In [ ]:
#Evaluate the frozen candidate on test data
test_features = candidate_scaler.transform(
    X_test[candidate_features]
)

mlp_test_predictions = candidate_model.forward(
    test_features
).ravel()

linear_test_predictions = candidate_linear.predict(
    test_features
)

test_results = pd.DataFrame({
    "HLS estimate": evaluate_predictions(
        y_test,
        X_test["lut"].to_numpy(),
    ),
    "Linear regression": evaluate_predictions(
        y_test,
        linear_test_predictions,
    ),
    "Custom MLP": evaluate_predictions(
        y_test,
        mlp_test_predictions,
    ),
}).T

display(test_results)